In [2]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import lightgbm as lgb
from lightgbm import LGBMRegressor

import time
set_config(display="text")  # displays text representation of estimators

In [3]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [4]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [5]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [6]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [7]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [8]:
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 7],
    'num_leaves': [31, 50, 100],
    'min_data_in_leaf': [10, 20, 50],
    'n_controls_per_case': [1, 2, 4, 10, 15]
}

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [9]:
# LightGBM Survival Model Wrapper
class LGBMSurvival:
    """
    LightGBM wrapper for survival analysis.
    Uses Cox proportional hazards loss approximation.
    """
    def __init__(self, learning_rate=0.1, n_estimators=100, max_depth=-1, 
                 num_leaves=31, min_data_in_leaf=20, random_state=None):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.num_leaves = num_leaves
        self.min_data_in_leaf = min_data_in_leaf
        self.random_state = random_state
        self.model = None
        
    def fit(self, X, y):
        """
        Fit LightGBM model.
        y: structured array with 'event' and 'time' fields
        """
        # Convert survival data to regression target
        # Use negative time for censored, positive for events
        y_train = np.where(y['event'], y['time'], -y['time'])
        
        self.model = LGBMRegressor(
            learning_rate=self.learning_rate,
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            num_leaves=self.num_leaves,
            min_data_in_leaf=self.min_data_in_leaf,
            random_state=self.random_state,
            objective='regression',
            verbose=-1
        )
        
        self.model.fit(X, y_train)
        return self
    
    def predict(self, X):
        """
        Predict risk scores (higher = higher risk)
        """
        return self.model.predict(X)
    
    def score(self, X, y):
        """
        Calculate concordance index
        """
        predictions = self.predict(X)
        return concordance_index_censored(y['event'], y['time'], predictions)[0]
    
    def predict_survival_function(self, X, times=None):
        """
        Approximate survival function using Breslow estimator
        """
        risk_scores = self.predict(X)
        # Simple approximation: S(t) = exp(-risk_score * baseline_hazard(t))
        # For compatibility, return callable functions
        surv_funcs = []
        for score in risk_scores:
            # Create a simple decreasing function
            def make_surv_func(s):
                return lambda t: np.exp(-np.abs(s) * t / 365.0)
            surv_funcs.append(make_surv_func(score))
        return surv_funcs

In [10]:
results_df, best_params, summary = generic_grid_search(
    LGBMSurvival,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='LightGBM',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1,
    dataset=subsampling,
    feature_columns=feature_columns
)


LIGHTGBM GRID SEARCH (Manual Implementation)
Model: LGBMSurvival
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - learning_rate: [0.01, 0.05, 0.1]
  - n_estimators: [100, 200, 500]
  - max_depth: [3, 5, 7]
  - num_leaves: [31, 50, 100]
  - min_data_in_leaf: [10, 20, 50]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total combinations: 1215

[1/1215] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3, 'num_leaves': 31, 'min_data_in_leaf': 10}, n_controls_per_case=1
  c-index: 0.6530 ± 0.1384
  c-index-ipcw: 0.5840 ± 0.0504
  int-brier-score: 0.5412 ± 0.0402
[2/1215] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3, 'num_leaves': 31, 'min_data_in_leaf': 10}, n_controls_per_case=2
  Elapsed: 0.1min, Est. remaining: 93.0min
  c-index: 0.5470 ± 0.1645
  c-index-ipcw: 0.5058 ± 0.1011
  int-brier-score: 0.4905 ± 0.0786
[3/1215] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, '

KeyboardInterrupt: 

In [ ]:
results_df.to_csv("../results/subsampling/lightgbm_subsampling_grid_search_results.csv", index=False)

Running the test on the best and correct parameters

In [ ]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/lightgbm_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  learning_rate: {best_params['learning_rate']}")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
print(f"  max_depth: {int(best_params['max_depth'])}")
print(f"  num_leaves: {int(best_params['num_leaves'])}")
print(f"  min_data_in_leaf: {int(best_params['min_data_in_leaf'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  learning_rate: 0.05
  n_estimators: 100
  max_depth: 1
  min_samples_split: 2
  n_controls_per_case: 1
  c-index-ipcw_mean (CV): 0.7469



In [ ]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['n_controls_per_case']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 54
Test samples: 594


In [ ]:
# Train final model with best hyperparameters
final_model = LGBMSurvival(
    learning_rate=best_params['learning_rate'],
    n_estimators=int(best_params['n_estimators']),
    max_depth=int(best_params['max_depth']),
    num_leaves=int(best_params['num_leaves']),
    min_data_in_leaf=int(best_params['min_data_in_leaf']),
    random_state=12
)

final_model.fit(x_train_sub, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [ ]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train_sub['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train_sub['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train_sub, y_test, preds_at_times, times)

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")

\nTest Set Performance:
  C-index: 0.6758
  C-index IPCW: 0.6381
  Integrated Brier Score: 0.0504
